In [ ]:
import torch
from torchvision import models, transforms
from PIL import Image

# Load and preprocess the image
def load_image(image_path):
    img = Image.open(image_path)
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    img = transform(img).unsqueeze(0)  # Add batch dimension
    return img

# Squeeze-and-Excitation Attention block
class SEBlock(torch.nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = torch.nn.AdaptiveAvgPool2d(1)
        self.fc = torch.nn.Sequential(
            torch.nn.Linear(channel, channel // reduction, bias=False),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(channel // reduction, channel, bias=False),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class MultiStageFeatureResNet(torch.nn.Module):
    def __init__(self, base_model='resnet50', weights='IMAGENET1K_V1', reduced_dim=64):
        super().__init__()
        weight_dict = {
            'resnet50': models.ResNet50_Weights,
            'resnet101': models.ResNet101_Weights,
            'resnet152': models.ResNet152_Weights
        }

        original_model = getattr(models, base_model)(weights=weight_dict[base_model][weights])
        self.conv1 = original_model.conv1
        self.bn1 = original_model.bn1
        self.relu = original_model.relu
        self.maxpool = original_model.maxpool
        self.layer1 = original_model.layer1
        self.layer2 = original_model.layer2
        self.layer3 = original_model.layer3
        self.layer4 = original_model.layer4

        # Adding SEBlocks
        self.se1 = SEBlock(256)
        self.se2 = SEBlock(512)
        self.se3 = SEBlock(1024)
        self.se4 = SEBlock(2048)

        # Additional convolutions for dimension matching
        self.reduce_dim1 = torch.nn.Conv2d(256, reduced_dim, 1)
        self.reduce_dim2 = torch.nn.Conv2d(512, reduced_dim, 1)
        self.reduce_dim3 = torch.nn.Conv2d(1024, reduced_dim, 1)
        self.reduce_dim4 = torch.nn.Conv2d(2048, reduced_dim, 1)

        # Adaptive pooling to match spatial dimensions
        self.adaptive_pool = torch.nn.AdaptiveAvgPool2d((3, 3))  # Pooling to 3x3

        # Final average pooling layer to reduce to 1x1 per channel
        self.avgpool = torch.nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        out1 = self.layer1(x)
        out1 = self.se1(out1)
        x1 = self.reduce_dim1(out1)
        x1 = self.adaptive_pool(x1)

        out2 = self.layer2(out1)
        out2 = self.se2(out2)
        x2 = self.reduce_dim2(out2)
        x2 = self.adaptive_pool(x2)
        fused2 = x1 + x2

        out3 = self.layer3(out2)
        out3 = self.se3(out3)
        x3 = self.reduce_dim3(out3)
        x3 = self.adaptive_pool(x3)
        fused3 = fused2 + x3

        out4 = self.layer4(out3)
        out4 = self.se4(out4)
        x4 = self.reduce_dim4(out4)
        x4 = self.adaptive_pool(x4)
        fused4 = fused3 + x4

        # Global average pooling to generate the final deep feature vector
        final_features = self.avgpool(fused4)
        final_features = torch.flatten(final_features, 1)

        return final_features

def main(image_path, base_model='resnet50', weights='IMAGENET1K_V1', reduced_dim=64):
    img = load_image(image_path)
    custom_model = MultiStageFeatureResNet(base_model=base_model, weights=weights, reduced_dim=reduced_dim)
    custom_model.eval()  # Set model to evaluation mode

    with torch.no_grad():
        deep_features = custom_model(img)

    feature_vector = deep_features.squeeze().tolist()  # Save to list

    return feature_vector

# # Extract
# feature_vector = main('Skin_Disease/1. Eczema 1677/0_0.jpg', base_model='resnet50', weights='IMAGENET1K_V1', reduced_dim=16)
# print("Length of final feature vector:", len(feature_vector))

In [ ]:
import os
import csv
from tqdm import tqdm  # Import tqdm for progress bar
import torch

def save_features_to_csv(root_dir, output_csv, base_model='resnet50', weights='IMAGENET1K_V1', reduced_dim=16):
    # Count the total number of images for tqdm initialization
    total_images = sum([len(files) for r, d, files in os.walk(root_dir)])

    with open(output_csv, 'w', newline='') as csvfile:
        csvwriter = csv.writer(csvfile)
        csvwriter.writerow(['image_name', 'label', 'relative_path', 'feature_vector'])

        # Initialize tqdm with the total number of images
        with tqdm(total=total_images, desc="Extracting features", unit="image") as pbar:
            for label in os.listdir(root_dir):
                class_dir = os.path.join(root_dir, label)
                if os.path.isdir(class_dir):
                    for img_name in os.listdir(class_dir):
                        img_path = os.path.join(class_dir, img_name)
                        relative_path = os.path.relpath(root_directory + "/" + img_path, root_dir)

                        # Replace backslashes with forward slashes for consistency
                        relative_path = relative_path.replace("\\", "/")

                        feature_vector = main(img_path, base_model, weights, reduced_dim)
                        feature_str = ' '.join(map(str, feature_vector))  # Format the feature vector as space-separated
                        csvwriter.writerow([img_name, label, relative_path, feature_str])

                        # Update progress bar after processing each image
                        pbar.update(1)

# Set root directory and CSV file path
root_directory = 'Skin_Disease'
output_csv_file = 'Skin_Diseases_resnet152_stage4_features_dim_1024.csv'

# Extract features and save to CSV
save_features_to_csv(root_directory, output_csv_file, base_model='resnet152', weights='IMAGENET1K_V1', reduced_dim=1024)
#1024
#512
#128
#64
#32
#16
print(f"Feature extraction completed. Saved to {output_csv_file}.")
